# Regressione delta_tas vs delta_cover (replica letterale)

Replica esatta dello schema di `06-covariance_albedo` + `06b-covariance_tas` +
`07-covariance_regression.ipynb` (analisi albedo-temperatura), applicato a
vegetation cover (cvh/cvl) al posto dell'albedo.

Per SENS e CTRL si calcola uno "skill vs osservazioni" (indice di correlazione
normalizzato, non letteralmente una covarianza, stesso nome usato nel codice
originale):

```
skill = (anomalia_modello * anomalia_obs) / (std_modello * std_obs)
```

- **Temperatura**: obs = ERA5 (come 06b), skill genuino per entrambi SENS e CTRL.
- **Cover**: obs-proxy = `cover_SENS` (per costruzione SENS e' forzato con le
  osservazioni). Lo skill di CTRL e' genuino (modello di vegetazione dinamico
  libero vs osservazioni); lo skill di SENS e' **circolare** (confronta le
  osservazioni con se stesse) — incluso comunque come richiesto, per confronto
  esplicito con `01-cover_tas_regression_adapted.ipynb`.

Poi `delta = skill_SENS - skill_CTRL` per entrambe le variabili, regredite tra
loro (mappa per pixel + box Siberia), per ciascuna delle 10 combinazioni di
lead year (annuale).

> **Nota**: a causa della circolarita' dello skill di SENS per la cover, il
> risultato di questa analisi va interpretato con cautela (vedi discussione
> nella sessione di lavoro). Il notebook `01-...adapted` propone un disegno
> alternativo non circolare.


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
era_var = '2t'
variables = ['cvh', 'cvl']
SAVE_PATH = str(FIG_DIR / "02_literal")  # sottocartella dedicata a questo notebook
os.makedirs(SAVE_PATH, exist_ok=True)


In [ ]:
# La logica di calcolo sta in un modulo importabile (cover_tas_lib.py): i
# processi spawn usati sotto (ogni task in un processo fresco) non vedono le
# funzioni definite nel notebook.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import run_one_literal, LEADS


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, era_var, y1, y2, SAVE_PATH) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_literal, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
